# Spark-Hive Showcase
Tutaj znajduje się przykładowy opis jak ustawić połączenie sparka z istniejącym metastore'm hive'a

In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__) # Ważne! 3.4.1

3.4.1


In [2]:
# Przykładowe połączenie Sparka z Hive'm:
# .enableHiveSupport() sprawia, że Spark:
# - ładuje hive-site.xml (jeśli jest dostępny)
# - korzysta z Hive Metastore
# - umożliwia DDL/DML Hive SQL (CREATE DATABASE, SHOW TABLES itp.)
spark = (
    SparkSession.builder
    .appName("spark-hive")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    # metastore.uris może być w hive-site.xml (i w sumie powinno być, ale nie chce mi się ruszać docker-compose już xd),
    # ale nadpisanie tutaj jest ok
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 15:30:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/24 15:30:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/24 15:30:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [18]:
# Wyświetl magazyn/ścieżkę baz danych (HDFS / lokalnie)
print("\n📦 Warehouse dir:", spark.conf.get("spark.sql.warehouse.dir"))

# Pokaż bazy danych Hive
print("---------Bazy Danych-------------")
spark.sql("SHOW DATABASES").show()

spark.sql("""
    CREATE DATABASE IF NOT EXISTS testdb
""")

print("Baza testdb istnieje")

# Zobaczmy metadane bazy testdb
print("---------Metadane testdb-------------")
spark.sql("DESCRIBE DATABASE testdb").show(truncate=False)

# Korzystamy z przykładowej bazy danych
spark.sql("USE testdb")

# Tworzenie tabeli, jeśli jej nie ma
spark.sql("""
    CREATE TABLE IF NOT EXISTS testdb.users (
        id INT,
        name STRING,
        age INT
    )
    STORED AS PARQUET
""")

# --- Wstawianie przykładowych danych, ale tylko gdy tabela jest pusta ---
count = spark.sql("SELECT COUNT(*) FROM testdb.users").first()[0]

if count == 0:
    print("\nTabela była pusta – dodaję dane...")
    spark.sql("""
        INSERT INTO testdb.users VALUES
        (1, 'Lukasz', 23),
        (2, 'Anna', 30),
        (3, 'Kuba', 27)
    """)
else:
    print("\nTabela posiada już dane – nie dodaję")

# Odczytujemy dane
print("----------SELECT z testdb.users------------")
df = spark.sql("SELECT * FROM users")
df.show()

# Sprawdzamy opis tabeli
print("---------Metadane testdb.users-------------")
spark.sql("DESCRIBE FORMATTED users").show(truncate=False)

spark.stop()

25/11/23 18:50:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/23 18:50:15 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic



📦 Warehouse dir: hdfs://namenode:8020/user/hive/warehouse
---------Bazy Danych-------------
+-----------+
|  namespace|
+-----------+
|    default|
|test_hive_2|
|     testdb|
+-----------+

Baza testdb istnieje
---------Metadane testdb-------------
+--------------+--------------------------------------------------+
|info_name     |info_value                                        |
+--------------+--------------------------------------------------+
|Catalog Name  |spark_catalog                                     |
|Namespace Name|testdb                                            |
|Comment       |                                                  |
|Location      |hdfs://namenode:8020/user/hive/warehouse/testdb.db|
|Owner         |hive                                              |
+--------------+--------------------------------------------------+



25/11/23 18:50:16 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                


Tabela posiada już dane – nie dodaję
----------SELECT z testdb.users------------
+---+------+---+
| id|  name|age|
+---+------+---+
|  1|Lukasz| 23|
|  2|  Anna| 30|
|  3|  Kuba| 27|
+---+------+---+

---------Metadane testdb.users-------------
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                              |comment|
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------+-------+
|id                          |int                                                                                                                                    |null   |
|name                        |string                  

In [3]:
print("-------------Bazy Danych-------------")
spark.sql("SHOW DATABASES").show()

-------------Bazy Danych-------------


25/11/24 15:30:07 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


+-----------------+
|        namespace|
+-----------------+
|cryptopredictions|
|          default|
|      test_hive_2|
|           testdb|
+-----------------+



In [5]:
print("-------------Bazy Danych-------------")
spark.sql("SHOW DATABASES").show()

spark.sql("""
        CREATE DATABASE IF NOT EXISTS CryptoPredictions
        COMMENT 'Database for crypto, stock index and currency exchange snapshots'""")
spark.sql("USE CryptoPredictions")

# ====================================================================================
# 1. CryptocurrencySnapshot
# ====================================================================================
spark.sql("""
CREATE TABLE IF NOT EXISTS CryptocurrencySnapshot (
    Symbol                 STRING,
    Datetime               TIMESTAMP,
    CurrentPrice           DOUBLE,
    OpeningPrice           DOUBLE,
    LowestDayPrice         DOUBLE,
    HighestDayPrice        DOUBLE,
    PreviousClosingPrice   DOUBLE
)
PARTITIONED BY (PartitionDate DATE)
STORED AS PARQUET
""")

# ====================================================================================
# 2. IndexSnapshot
# ====================================================================================
spark.sql("""
CREATE TABLE IF NOT EXISTS IndexSnapshot (
    IndexName                    STRING,
    Datetime                     TIMESTAMP,
    CurrentPrice                 DOUBLE,
    CurrentVolume                BIGINT,
    OpeningPrice                 DOUBLE,
    LowestDayPrice               DOUBLE,
    HighestDayPrice              DOUBLE,
    LowestYearlyPrice            DOUBLE,
    HighestYearlyPrice           DOUBLE,
    FiftyDayAveragePrice         DOUBLE,
    TwoHundredDaysAveragePrice   DOUBLE,
    TenDayAverageVolume          BIGINT,
    ThreeMonthAverageVolume      BIGINT,
    YearOverYearPriceChange      DOUBLE
)
PARTITIONED BY (PartitionDate DATE)
STORED AS PARQUET
""")

# ====================================================================================
# 3. USDExchangeRates
# ====================================================================================
spark.sql("""
CREATE TABLE IF NOT EXISTS USDExchangeRates (
    CurrencyTo            STRING,
    Date                  DATE,
    NumberOfTrades        BIGINT,
    VolumeTraded          DOUBLE,
    OpeningPrice          DOUBLE,
    ClosingPrice          DOUBLE,
    AveragedPrice         DOUBLE,
    LowestDailyPrice      DOUBLE,
    HighestDailyPrice     DOUBLE
)
PARTITIONED BY (PartitionDate DATE)
STORED AS PARQUET
""")

spark.sql("SHOW TABLES").show()

print("All Hive tables created successfully.")

-------------Bazy Danych-------------
+-----------------+
|        namespace|
+-----------------+
|cryptopredictions|
|          default|
|      test_hive_2|
|           testdb|
+-----------------+

+-----------------+--------------------+-----------+
|        namespace|           tableName|isTemporary|
+-----------------+--------------------+-----------+
|cryptopredictions|cryptocurrencysna...|      false|
|cryptopredictions|    usdexchangerates|      false|
|cryptopredictions|       indexsnapshot|      false|
+-----------------+--------------------+-----------+

All Hive tables created successfully.


In [11]:
spark.stop()